# Introduction to Neuroinformatics

## Exercise session 8: Dynamical systems

Welcome to the eighth exercise session of the introduction to neuroinformatics course. In the previous session, we used dimensionality reduction to describe patterns of population activity. Here, we will study how activity changes over time using dynamical systems.

We will start with one- and two-dimensional linear systems, then explore eigenvalues, noisy observations, low-dimensional manifolds and nonlinear dynamics. These methods will prepare us to study recurrent neural networks in the next session.

This notebook is designed to be run on Google Colab. If you wish to save your progress, click on the "Copy to Drive" button.

---

# Table of contents

* [Packages](#packages)
* [1: One-dimensional linear systems](#1)
* [2: Two-dimensional linear systems and flow fields](#2)
* [3: Modes of linear systems](#3)
  * [3.1: Real and imaginary parts of eigenvalues](#3.1)
* [4: Noise and fitting dynamics](#4)
* [5: High-dimensional dynamical systems and manifolds](#5)
  * [5.1: Finding a low-dimensional subspace with PCA](#5.1)
* [6: Nonlinear dynamics and linearization](#6)
  * [6.1: Fixed points and slow points](#6.1)
  * [6.2: Local linearization](#6.2)
* [7: Further Readings](#further)

---

# Packages <a name="packages"></a>

Run the following cells to download the helper module and figure, then import the libraries used in this exercise. The setup cell runs only on Google Colab, the platform supported by this course. NumPy, Matplotlib, SciPy and ipywidgets are already available there; the setup also ensures that Plotly is installed for the rotatable three-dimensional plot.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"IN_COLAB: {IN_COLAB}")

In [ ]:
if IN_COLAB:
    !pip -q install plotly
    !mkdir -p utils_ex8
    !mkdir -p utils_ex8/figures

    # -O rather than -P for the code and figures: wget -P leaves an existing file
    # alone and writes name.1 beside it, so a corrected helper would never reach a
    # runtime that has run this cell before. This notebook fetches no bulk data.
    !wget -O utils_ex8/dynamical_systems_utils.py https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex8/dynamical_systems_utils.py
    !wget -O utils_ex8/figures/gallego_manifold.png https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex8/figures/gallego_manifold.png

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
from ipywidgets import Button, FloatSlider, VBox, interact, interactive

from utils_ex8.dynamical_systems_utils import (
    Noisy3DExplorer,
    compare_dynamics_and_search,
    compute_jacobian,
    continuous_update,
    fit_dynamics_matrix_continuous,
    interactive_eigval_plot,
    plot_2d_flowfield,
    simulate_noisy_system,
)

---

# 1: One-dimensional linear systems <a name="1"></a>

An ordinary differential equation (ODE) describes how a state changes over time. For a one-dimensional state $x$, we write $\dot{x} = f(x)$, where $\dot{x} = dx/dt$. To obtain a trajectory, we also need an initial condition, $x(0)$.

We will start with two linear systems:

1. $\dot{x} = 1.5x$, with $x(0) = 1$.
2. $\dot{x} = -1.5x$, with $x(0) = 1$.

These equations describe continuous time. To simulate them on a computer, we will use **Euler's method**: $x(t + \Delta t) \approx x(t) + \Delta t\,f(x(t))$. A small time step approximates the continuous trajectory. We will use arbitrary time units throughout the simulations.

In [ ]:
t0 = 0
tf = 1
dt = 0.01
t = np.arange(t0, tf, dt)

x1 = np.zeros_like(t)
x2 = np.zeros_like(t)
x1[0] = 1
x2[0] = 1

for i in range(1, len(t)):
    x1[i] = x1[i - 1] + dt * (1.5 * x1[i - 1])
    x2[i] = x2[i - 1] + dt * (-1.5 * x2[i - 1])

plt.figure(figsize=(10, 6))
plt.plot(t, x1, label=r"$\dot{x} = 1.5x, x(0) = 1$", color="b")
plt.plot(t, x2, label=r"$\dot{x} = -1.5x, x(0) = 1$", color="r")
plt.title("One-dimensional linear systems")
plt.xlabel("Time (arbitrary units)")
plt.ylabel("x(t)")
plt.legend()
plt.show()

---

> **Assignment 1**
>
> Which system is attracting and which is unstable? Explain your answer using the equations and the trajectories above.
>
> <details>
>
> <summary>Solution</summary>
>
> For $\dot{x} = 1.5x$, positive states grow and negative states become more negative. The origin is a fixed point, but it is unstable: a small displacement grows over time. For $\dot{x} = -1.5x$, trajectories from either side approach the origin, so it is attracting. The plotted positive initial condition illustrates each case; the equations tell us what happens for other starting states.
> </details>

> **Assignment 2**
>
> Can you write a one-dimensional equation whose derivative is positive for every state, so that every trajectory increases wherever the solution exists?
>
> <details>
>
> <summary>Solution</summary>
>
> One example is $\dot{x} = 1 + x^2$. Its derivative is strictly positive for every real $x$, so trajectories increase. This does not imply that solutions remain finite for all time.
> </details>

---

A single trajectory shows what happens from one initial condition. To understand the behaviour of the system more generally, we can repeat the simulation from several starting points. Run the next cell and compare the two sets of trajectories.

In [ ]:
t0 = 0
tf = 5
dt = 0.01
t = np.arange(t0, tf, dt)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
starts = np.linspace(-5, 5, 20)

for start in starts:
    x1 = np.zeros_like(t)
    x2 = np.zeros_like(t)
    x1[0] = start
    x2[0] = start
    for i in range(1, len(t)):
        x1[i] = x1[i - 1] + dt * (1.5 * x1[i - 1])
        x2[i] = x2[i - 1] + dt * (-1.5 * x2[i - 1])
    axes[0].plot(t, x1, color="b")
    axes[1].plot(t, x2, color="r")

axes[0].set_title(r"$\dot{x} = 1.5x$")
axes[1].set_title(r"$\dot{x} = -1.5x$")
for ax in axes:
    ax.set_xlabel("Time (arbitrary units)")
    ax.set_ylabel("x(t)")
plt.show()

Both examples are linear. In the general equation $\dot{x} = f(x)$, however, $f$ can be nonlinear. Later in this session, we will see how to study a nonlinear system by examining its behaviour near fixed points.

---

# 2: Two-dimensional linear systems and flow fields <a name="2"></a>

We can extend the same idea to a state with two coordinates, $\mathbf{x} = (x, y)^T$:

$$
\begin{pmatrix}\dot{x} \\ \dot{y}\end{pmatrix}
= A\begin{pmatrix}x \\ y\end{pmatrix}.
$$

The **dynamics matrix** $A$ determines how each coordinate changes. In the next cell, we will simulate a system from two initial conditions. Add a few more starting points, including points on the coordinate axes, to explore its behaviour.

In [ ]:
initial_conditions = [(1, 1), (1, 2)]
A = np.array([[1, 0], [0, -1]])

plt.figure()
for z0 in initial_conditions:
    z = np.zeros((len(t), 2))
    z[0] = z0
    for i in range(1, len(t)):
        z[i] = z[i - 1] + dt * (A @ z[i - 1])
    plt.plot(z[:, 0], z[:, 1], label=f"Initial condition: {z0}")

plt.title("Two-dimensional linear system")
plt.xlabel("x")
plt.ylabel("y")
plt.axhline(0, color="black", linewidth=0.5)
plt.axvline(0, color="black", linewidth=0.5)
plt.legend()
plt.show()

---

> **Assignment 3**
>
> How would you classify the origin of this system? Describe what happens along each coordinate axis and for a starting point away from both axes.
>
> <details>
>
> <summary>Solution</summary>
>
> The origin is an unstable saddle point. Along the $y$ axis, trajectories approach the origin because $\dot{y} = -y$. Along the $x$ axis, they move away because $\dot{x} = x$. A trajectory with a nonzero $x$ coordinate eventually moves away, even while its $y$ coordinate decays.
> </details>

---

Rather than simulating every possible starting point, we can draw a **flow field** in state space. Each arrow shows the derivative at one location:

1. Its direction shows the direction of change.
2. Its length represents the speed of change, $\|\dot{\mathbf{x}}\|$.

The arrows describe local motion; trajectories follow this field over time.

In [ ]:
A = np.array([[1, 0], [0, -1]])


def linear_system(state):
    return A @ state


plot_2d_flowfield(linear_system, title="Two-dimensional flow field")
plt.show()

---

> **Assignment 4**
>
> In the flow field above, how can you recognise a fixed point? Does the existence of a fixed point guarantee that it is stable?
>
> <details>
>
> <summary>Solution</summary>
>
> At the origin, both derivatives are zero, so the arrow has zero length. A trajectory starting exactly there stays there. This identifies a fixed point, but does not establish stability: nearby trajectories can move away. Here, the origin is an unstable saddle.
> </details>

---

---

# 3: Modes of linear systems <a name="3"></a>

A flow field helps us recognise patterns, but small arrows alone do not tell us whether trajectories approach or leave a fixed point. For a linear system, we can answer this using the **eigenvalues and eigenvectors** of $A$.

An eigenvector gives a mode of the system. For $\dot{\mathbf{x}} = A\mathbf{x}$, its amplitude changes as $e^{\lambda t}$, where $\lambda$ is the corresponding eigenvalue. When $A$ has a full set of independent eigenvectors, we can write $A = Q\Lambda Q^{-1}$: the columns of $Q$ are the eigenvectors, and the diagonal of $\Lambda$ contains their eigenvalues.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(A)

Eigenvalues can have real and imaginary parts. We will plot them in the complex plane, with the real part on the horizontal axis and the imaginary part on the vertical axis. For our current matrix, both eigenvalues are real.

For these continuous-time systems, the sign of an eigenvalue's real part determines whether its mode grows or decays. The vertical line at zero separates the two cases.

In [ ]:
plt.figure(figsize=(4, 4))
plt.axvline(0, color="grey", linewidth=1)
plt.axhline(0, color="grey", linewidth=1)
plt.scatter(eigenvalues.real, eigenvalues.imag, label="Eigenvalues")
plt.xlabel("Real part")
plt.ylabel("Imaginary part")
plt.title("Eigenvalues of A")
plt.xlim(-2, 2)
plt.ylim(-2, 2)
plt.gca().set_aspect("equal")
plt.legend()
plt.show()

The eigenvalue $+1$ corresponds to growth along the $x$ direction, while $-1$ corresponds to decay along the $y$ direction. The origin is therefore a **saddle point**: it attracts along one direction and repels along another.

---

## 3.1: Real and imaginary parts of eigenvalues <a name="3.1"></a>

Nonzero imaginary parts describe oscillatory motion. For a real matrix, non-real eigenvalues occur in complex-conjugate pairs. The next widget lets you change the four entries of $A$ and a constant input $b$, added to both coordinates in $\dot{\mathbf{x}} = A\mathbf{x} + (b, b)^T$. The left panel shows the eigenvalues computed from that matrix; the right panel shows its flow field.

Use the sliders to explore different matrices, then follow the settings in Assignments 5 and 6. On Colab, the plot updates when you release a slider.

In [ ]:
_ = interact(
    interactive_eigval_plot,
    a11=FloatSlider(min=-2, max=2, step=0.1, value=1, description="a₁₁", continuous_update=continuous_update),
    a12=FloatSlider(min=-2, max=2, step=0.1, value=0, description="a₁₂", continuous_update=continuous_update),
    a21=FloatSlider(min=-2, max=2, step=0.1, value=0, description="a₂₁", continuous_update=continuous_update),
    a22=FloatSlider(min=-2, max=2, step=0.1, value=-1, description="a₂₂", continuous_update=continuous_update),
    b=FloatSlider(min=-2, max=2, step=0.1, value=0, description="b (both axes)", continuous_update=continuous_update),
)

---

> **Assignment 5**
>
> Set both diagonal entries to zero, with $a_{12} = -1$, $a_{21} = 1$ and $b = 0$.
>
> 1. What motion do you see, and how does it relate to the imaginary parts of the eigenvalues? Reverse the signs of both off-diagonal entries and compare.
> 2. Keep the off-diagonal entries opposite in sign and equal in magnitude. Set both diagonal entries to the same negative value, then to the same positive value. What changes?
>
> <details>
>
> <summary>Solution</summary>
>
> The first setting produces counter-clockwise rotation with eigenvalues $+i$ and $-i$. Reversing both off-diagonal signs reverses the rotation, but the eigenvalue pair is unchanged: the pair alone does not specify its direction. A common diagonal value $a$ gives eigenvalues $a \pm i$. For negative $a$, trajectories spiral inward; for positive $a$, they spiral outward.
> </details>

> **Assignment 6**
>
> Set $a_{11} = 0$, $a_{22} = -1$, both off-diagonal entries to zero and $b = 0$. What does the zero eigenvalue imply about motion along its eigenvector?
>
> <details>
>
> <summary>Solution</summary>
>
> The corresponding coordinate stays constant: it neither grows nor decays. Here, $\dot{x} = 0$ and $\dot{y} = -y$, so every point on the $x$ axis is a fixed point, and trajectories approach that axis. The zero mode is neutral, rather than attracting. A zero eigenvalue alone does not guarantee stability for a general system.
> </details>

---

---

# 4: Noise and fitting dynamics <a name="4"></a>

In an experiment, we observe neural activity over time without knowing the equations that generated it. A small dataset might look like this, with firing rates in arbitrary units:

| Time (ms) | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| Neuron 1 firing rate | 1 | 0.5 | 0.3 | 1 | 0.8 |
| Neuron 2 firing rate | 0.3 | 0.6 | 0.5 | 0.9 | 1 |

To estimate linear dynamics, we first fit a **discrete-time transition matrix** $B$ that predicts the next observed state:

$$
\underset{B}{\operatorname{argmin}}\sum_t
\|\mathbf{x}(t + \Delta t) - B\mathbf{x}(t)\|^2.
$$

For observations of a continuous linear system, $B = e^{A\Delta t}$. The helper estimates $B$ by least squares, then computes $A = \log(B)/\Delta t$ using the matrix logarithm. Recovering the underlying dynamics requires sufficiently informative data; noise, limited sampling and model mismatch can affect the estimate.

The next cell generates four noisy trajectories with the same saddle structure as before, using smaller coefficients. The simulator takes one Euler step per arbitrary time unit, so we pass $\Delta t = 1$ to the fitter. We will compare the fitted flow field with the observed trajectories.

In [ ]:
def explore_noisy_fit(noise_level):
    global noisy_data, A_fitted
    noisy_data = simulate_noisy_system(noise_level=noise_level)
    A_fitted = fit_dynamics_matrix_continuous(noisy_data, dt=1.0)

    def fitted_system(state):
        return state @ A_fitted

    plot_2d_flowfield(
        fitted_system,
        data=noisy_data,
        title=f"Fitted linear system, noise level: {noise_level:.2f}",
    )
    plt.show()


noise_fit_widget = interactive(
    explore_noisy_fit,
    noise_level=FloatSlider(
        min=0, max=0.5, step=0.01, value=0.01,
        description="Noise level", continuous_update=False,
    ),
)
resample_fit = Button(description="New sample")
resample_fit.on_click(lambda button: noise_fit_widget.update())
display(VBox([noise_fit_widget, resample_fit]))

---

> **Assignment 7**
>
> Increase the noise slider above to rerun the simulation and fit. Describe two effects on the trajectories or fitted flow field. Use **New sample** several times at the same noise level to assess how consistent the fit is.
>
> <details>
>
> <summary>Solution</summary>
>
> The trajectories become more irregular, and the estimated flow field can differ more from the underlying saddle dynamics. The fitted matrix also varies between runs because each simulation uses new noise. The effect depends on both the noise level and how well the trajectories sample state space.
> </details>

> **Assignment 8**
>
> What changes to data collection or analysis could reduce the uncertainty you observed in Assignment 7?
>
> <details>
>
> <summary>Solution</summary>
>
> Collect more trajectories, sample a wider range of initial conditions, or use a model that accounts for noise. Smoothing can help in some settings, but can also remove fast dynamics or introduce bias, so its effect should be checked.
> </details>

---

---

# 5: High-dimensional dynamical systems and manifolds <a name="5"></a>

So far, our states have had at most two coordinates. A neural recording can contain hundreds or thousands of neurons, with one state-space axis per neuron. For example:

| Time (ms) | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| Neuron 1 firing rate | 1 | 0.5 | 0.3 | 1 | 0.8 |
| … | … | … | … | … | … |
| Neuron 1020 firing rate | 0.3 | 0.6 | 0.5 | 0.9 | 1 |

Visualising a flow field in this space is difficult. However, neural activity often occupies a **low-dimensional manifold** within the full state space. The figure below illustrates this idea for three neurons.

In [ ]:
# Colab's markdown renderer cannot resolve relative paths to files on the VM,
# so the figure is displayed from Python instead.
display(Image("utils_ex8/figures/gallego_manifold.png", width=304))

<em>A two-dimensional neural manifold embedded in a three-dimensional activity space.</em>

Each axis (N1, N2, N3) represents one neuron's activity. The trajectories occupy a plane spanned by U1 and U2, rather than the full three-dimensional space. A manifold need not be flat; this plane is a simple example.

---

## 5.1: Finding a low-dimensional subspace with PCA <a name="5.1"></a>

In exercise session 7 (Population Representations), we used **Principal Component Analysis (PCA)** to quantify the dimensionality of population activity. Here, we will use it to find a low-dimensional description of trajectories. PCA finds orthogonal directions that capture the most variance after centring the data.

We will simulate a noisy three-dimensional linear system with 50 trajectories, then ask how many principal components capture its main structure.

In [ ]:
population = Noisy3DExplorer(noise_level=0.1)
display(population.widget)

Drag the three-dimensional plot to rotate it, and use the scroll wheel or toolbar to zoom. The noise slider updates when you release it; **New sample** generates another realisation at the same noise level.

Let us now examine how much variance each principal component explains. Once displayed, both PCA views below refresh automatically when you change the noise level or request a new sample above.

In [ ]:
display(population.variance_view())

---

> **Assignment 9**
>
> How much variance do the first two principal components explain together? How many components would you retain to describe most of these data?
>
> <details>
>
> <summary>Solution</summary>
>
> At the default noise level, the first two components usually explain nearly all the variance, so a two-dimensional description captures the main structure. The exact fractions vary between simulations; in some runs the first component alone may exceed 95%. A variance threshold is a practical summary, rather than an exact count of the underlying dynamical modes.
> </details>

> **Assignment 10**
>
> What does the pattern of explained variance suggest about the geometry of the trajectories in three-dimensional state space?
>
> <details>
>
> <summary>Solution</summary>
>
> Two principal-component directions span a plane. The trajectories lie approximately near this plane, with a much smaller spread in the remaining direction. PCA identifies a linear subspace; it does not establish that an arbitrary curved manifold is planar.
> </details>

> **Assignment 11**
>
> Why does the third principal component explain a small amount of variance rather than exactly zero?
>
> <details>
>
> <summary>Solution</summary>
>
> The simulation adds noise in all three coordinates, including the direction outside the plane of the noiseless trajectories. This produces nonzero variance in the third component.
> </details>

> **Assignment 12**
>
> What do you expect to happen to the fraction of variance explained by each component as the noise level increases? Use the noise slider above the three-dimensional plot, then compare several runs with **New sample**.
>
> <details>
>
> <summary>Solution</summary>
>
> Noise adds variation outside the main plane as well as within it, so the third component generally explains a larger fraction and the dominant component a smaller fraction. The fractions need not become equal: the underlying dynamics amplify some directions more strongly than others, and each run has a different noise realisation.
> </details>

---

Next, we will project the current trajectories onto the first two principal-component directions. Dots mark the initial states and crosses mark the final states. We keep the original origin visible to help interpret the direction of motion.

This view updates with the three-dimensional plot. The current observations and projected trajectories are available as `population.data` and `population.projection` if you want to analyse them further.

In [ ]:
display(population.projection_view())

---

> **Assignment 13**
>
> Use the initial and final markers in the projected trajectories to determine whether motion in the main plane is predominantly towards or away from the origin. Is the system attracting there?
>
> <details>
>
> <summary>Solution</summary>
>
> The trajectories move predominantly away from the origin, so the system is expanding in the main plane. The simulator has two positive eigenvalues and one zero eigenvalue. Low-dimensional structure alone does not imply attraction.
> </details>

> **Assignment 14**
>
> How could you check your answer to Assignment 13 quantitatively?
>
> <details>
>
> <summary>Solution</summary>
>
> Fit a linear dynamics matrix to the two-dimensional projected trajectories and examine its eigenvalues. Positive real parts support expansion; negative real parts would support attraction. For this simulation, the two growing modes have positive eigenvalues. Compare the result across several noise realisations.
> </details>

---

---

# 6: Nonlinear dynamics and linearization <a name="6"></a>

A linear system has the same dynamics matrix everywhere. A nonlinear system can behave differently in different parts of state space. We will explore this using

$$
\dot{x} = 3x - 2x^3, \qquad \dot{y} = -y.
$$

Run the next cell and look for locations where both derivatives vanish. We will then study the dynamics near each of these fixed points.

In [ ]:
def nonlinear_system(state):
    x, y = state
    return np.array([3 * x - 2 * x**3, -y])


plot_2d_flowfield(
    nonlinear_system,
    space=np.linspace(-2, 2, 15),
    title="Flow field of the nonlinear system",
)
plt.show()

---

## 6.1: Fixed points and slow points <a name="6.1"></a>

At a **fixed point** $\mathbf{x}^*$, $f(\mathbf{x}^*) = \mathbf{0}$, so a trajectory starting exactly there does not move. Near a **slow point**, the derivative is small but need not be zero.

We will compare two ways of moving from the same initial states:

1. **Forward dynamics** follows $\dot{\mathbf{x}} = f(\mathbf{x})$. In this example, starting states with nonzero $x$ approach one of the outer attracting fixed points.
2. **Fixed-point search** tries to reduce $q(\mathbf{x}) = \tfrac{1}{2}\|f(\mathbf{x})\|^2$. Its gradient is $\nabla q = J(\mathbf{x})^T f(\mathbf{x})$, where $J$ is the Jacobian (the matrix of partial derivatives, discussed in Section 6.2). The search follows a numerical descent direction, which can point against the system's dynamics and can lead to an unstable fixed point.

The search uses gradient descent with backtracking and stays within the box $[-2, 2]^2$. A continuous objective has a minimum on this compact box, but that does not guarantee that the algorithm finds a fixed point. We accept an endpoint only when $\|f(\mathbf{x})\|$ is small; a nonzero-residual endpoint is marked separately.

The two panels below use identical starting points and matching trajectory colours. Faint lines and arrows show forward motion on the left and accepted search steps on the right. Grey background arrows show the physical vector field in both panels. Search steps are iterations, not elapsed physical time.

In [ ]:
initial_states = np.array([
    (x, y)
    for x in [-1.3, -0.25, 0.25, 1.3]
    for y in [-1.5, -0.75, 0, 0.75, 1.5]
])

In [ ]:
comparison = compare_dynamics_and_search(nonlinear_system, initial_states)
unique_fixed_points = np.unique(comparison["fixed_points"].round(decimals=2), axis=0)

---

## 6.2: Local linearization <a name="6.2"></a>

We can approximate a smooth nonlinear system near a fixed point using its **Jacobian**, the matrix of partial derivatives of $f$. Define a small displacement $\delta\mathbf{x} = \mathbf{x} - \mathbf{x}^*$. A first-order Taylor expansion gives

$$
\dot{\delta\mathbf{x}} \approx J(\mathbf{x}^*)\,\delta\mathbf{x}.
$$

This approximation describes perturbations around the fixed point. As in Section 3, eigenvalues with negative real parts describe decaying modes, while a positive real part indicates an unstable mode. If an eigenvalue has zero real part, linearization alone may be inconclusive for a nonlinear system.

In [ ]:
for point in unique_fixed_points:
    print("Found fixed point:", point)

Copy one of the fixed points printed above into the next cell. Repeat for the other points and compare the eigenvalues of their Jacobians.

In [ ]:
# Replace this point with each of the fixed points printed above.
FIXED_POINT = [-1.22, 0]

jacobian = compute_jacobian(nonlinear_system, np.array(FIXED_POINT))
local_eigenvalues = np.linalg.eigvals(jacobian)

plt.figure(figsize=(6, 4))
plt.axvline(0, color="grey", linewidth=1)
plt.axhline(0, color="grey", linewidth=1)
plt.scatter(local_eigenvalues.real, local_eigenvalues.imag, label="Eigenvalues")
plt.xlabel("Real part")
plt.ylabel("Imaginary part")
plt.title(f"Jacobian eigenvalues near {FIXED_POINT}")
plt.legend()
plt.show()

---

> **Assignment 15**
>
> Compare the Jacobian eigenvalues at the three fixed points.
>
> 1. Which fixed point has a different stability pattern from the other two?
> 2. How do the two sets of paths in Section 6.1 illustrate this difference?
>
> <details>
>
> <summary>Solution</summary>
>
> The origin, $(0, 0)$, has one positive and one negative eigenvalue. The two outer fixed points have two negative eigenvalues. Forward trajectories from the displayed starts approach the outer points; the search can also reach the unstable origin, because it decreases the field norm rather than following the flow.
> </details>

> **Assignment 16**
>
> What kind of local dynamics do the eigenvalues predict near the origin? Relate them to the flow field.
>
> <details>
>
> <summary>Solution</summary>
>
> At the origin, the Jacobian is $\operatorname{diag}(3, -1)$. Motion grows along the $x$ direction and decays along the $y$ direction, giving an unstable saddle. Perturbations with a nonzero $x$ component move towards one of the outer attracting fixed points.
> </details>

> **Assignment 17**
>
> What kind of local dynamics occur near the other two fixed points? Explain using their Jacobian eigenvalues.
>
> <details>
>
> <summary>Solution</summary>
>
> The outer fixed points are $(\pm\sqrt{3/2}, 0)$, approximately $(\pm 1.22, 0)$. At either point, the Jacobian has eigenvalues $-6$ and $-1$, so small perturbations decay in both directions. Both fixed points are attracting. Using rounded coordinates in the numerical Jacobian gives values close to these exact eigenvalues.
> </details>

---

> **Bonus Assignment (next exercise session)**
>
> What kind of model could learn the dynamics of a nonlinear, high-dimensional system from sequences of observations? This question connects to the next exercise session.
>
> <details>
>
> <summary>Solution</summary>
>
> A recurrent neural network can learn nonlinear state transitions from sequential data. It is itself a nonlinear dynamical system, whose learned dynamics can be studied using the methods introduced here.
> </details>

---

# 7. Further Readings <a name="further"></a>

These resources expand on the methods used in this exercise session:

- Steven H. Strogatz, [*Nonlinear Dynamics and Chaos*](https://www.routledge.com/link/link/p/book/9781032707891). An introduction to flows, fixed points, linear systems and local stability analysis.
- Gallego, Perich, Miller & Solla (2017), ["Neural manifolds for the control of movement"](https://doi.org/10.1016/j.neuron.2017.05.025), *Neuron*. A connection between low-dimensional population activity and the neural control of movement.
- Cosma Shalizi, ["Principal Components Analysis II"](https://www.stat.cmu.edu/~cshalizi/dst/20/lectures/06/lecture-06.html). Lecture notes on principal-component directions, projections and explained variance.

**End of this exercise session.**